In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *
from biked_commons.conditioning import conditioning

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\benchmark_models\generative_models\../../..\biked_commons\prediction\usability_predictors.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malici

In [2]:
def sample_continuous(num_samples, split="test", randomize = False):
    emb = conditioning.sample_image_embedding(num_samples, split, randomize)
    rider = conditioning.sample_riders(num_samples, split, randomize)
    use_case = conditioning.sample_use_case(num_samples, split, randomize)
    all = torch.cat((emb, rider, use_case), dim=1)
    return all

def parse_continuous_condition(condition):
    image_embeddings = condition[:, :512]
    use_case_condition = condition[:, -3:]
    rider_condition = condition[:, 512:-3]
    condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
    return condition


In [3]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)


In [ ]:
def get_composite_score(constrant_vs_objective_weight == 10.0):
    data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)
    evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)

    isobjective = torch.tensor(requirement_types) == 1

    weights = get_ref_point(evaluator, requirement_names, requirement_names, reduction="meanabs")

    assert weights.min() > 0, "Ref point should be greater than 0"

    def calc_composite_score(data_tens, condition, evaluator = evaluator):
        eval_scores = evaluator(data_tens, condition)
        scaled_scores = eval_scores / weights

        objective_scores = scaled_scores[:, isobjective]
        constraint_scores = scaled_scores[:, ~isobjective]

        total_scores = torch.sum(objective_scores, dim=1) + torch.sum(constraint_scores, dim=1) * constrant_vs_objective_weight
        composite_scores = total_scores / (len(objective_scores) + len(constraint_scores) * constrant_vs_objective_weight)
        return composite_scores


    

    